In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 7 - Week 7
# --------------------------------------------------
# Use all accumulated observations stored in the Week 7 .npy files.
# Fit an ARD Matern GP, derive search widths from fitted lengthscales,
# generate local + wider + global candidates, then calibrate EI and UCB.

In [2]:
X = np.load("function7/initial_inputs.npy")
Y = np.load("function7/initial_outputs.npy").reshape(-1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

assert len(X) == len(Y)
assert X.shape[1] == 6

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:", best_x)
print("Current best observed output:", best_y)

X shape: (36, 6)
Y shape: (36,)

Current best observed input: [0.093684 0.325604 0.371379 0.230467 0.28146  0.638895]
Current best observed output: 2.66033631734043


In [3]:
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.ones(6) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
0.639**2 * Matern(length_scale=[0.309, 2, 2, 0.155, 0.148, 2], nu=2.5) + WhiteKernel(noise_level=1e-08)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co

In [4]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1 / lengthscales
sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:", lengthscales)
print("Normalised inverse-lengthscale sensitivity:", sensitivity)


ARD lengthscales: [0.30913701 2.         2.         0.15493131 0.14843551 2.        ]
Normalised inverse-lengthscale sensitivity: [0.18045144 0.02789211 0.02789211 0.36005774 0.3758145  0.02789211]


In [5]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal search widths:", local_scale)
print("Wider search widths:", wide_scale)


Local search widths: [0.07728425 0.1        0.1        0.03873283 0.03710888 0.1       ]
Wider search widths: [0.1545685  0.2        0.2        0.07746565 0.07421776 0.2       ]


In [6]:
rng = np.random.default_rng(42)

local_candidates = best_x + rng.normal(
    0,
    local_scale,
    size=(60000, 6)
)

wide_candidates = best_x + rng.normal(
    0,
    wide_scale,
    size=(30000, 6)
)

global_candidates = rng.uniform(
    0,
    1,
    size=(15000, 6)
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

candidates = np.clip(candidates, 0, 1)

print("Generated candidates:", len(candidates))

Generated candidates: 105000


In [7]:
tree = cKDTree(X)

distance, _ = tree.query(candidates, k=1)

candidates = candidates[distance > 0.008]

print("Candidates after filtering:", len(candidates))

Candidates after filtering: 105000


In [8]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

In [9]:
def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        + sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [10]:
print("\nEI calibration:\n")

for xi in [0.0, 0.001, 0.005, 0.01, 0.02]:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        f"xi={xi}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI calibration:

xi=0.0 
 candidate = [0.12877302 0.36524866 0.33262029 0.24929573 0.28037096 0.46875406] 
 mean = 2.700599 
 std = 0.070154 
 EI = 0.0526054 

xi=0.001 
 candidate = [0.12877302 0.36524866 0.33262029 0.24929573 0.28037096 0.46875406] 
 mean = 2.700599 
 std = 0.070154 
 EI = 0.05189083 

xi=0.005 
 candidate = [0.12877302 0.36524866 0.33262029 0.24929573 0.28037096 0.46875406] 
 mean = 2.700599 
 std = 0.070154 
 EI = 0.04908155 

xi=0.01 
 candidate = [0.12877302 0.36524866 0.33262029 0.24929573 0.28037096 0.46875406] 
 mean = 2.700599 
 std = 0.070154 
 EI = 0.04568296 

xi=0.02 
 candidate = [0.12877302 0.36524866 0.33262029 0.24929573 0.28037096 0.46875406] 
 mean = 2.700599 
 std = 0.070154 
 EI = 0.03927796 



In [11]:
print("\nUCB calibration:\n")

for beta in [0.1, 0.25, 0.5, 1.0, 1.5]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB calibration:

beta=0.1 
 candidate = [0.12158283 0.28741647 0.32312591 0.23681484 0.27857032 0.496277  ] 
 mean = 2.708019 
 std = 0.04244 
 UCB = 2.712263 

beta=0.25 
 candidate = [0.13058831 0.26788231 0.28691274 0.244098   0.27999812 0.53765723] 
 mean = 2.703638 
 std = 0.062077 
 UCB = 2.719157 

beta=0.5 
 candidate = [0.12877302 0.36524866 0.33262029 0.24929573 0.28037096 0.46875406] 
 mean = 2.700599 
 std = 0.070154 
 UCB = 2.735676 

beta=1.0 
 candidate = [0.14514087 0.16997591 0.29876258 0.24921097 0.28231768 0.36282021] 
 mean = 2.667101 
 std = 0.109382 
 UCB = 2.776483 

beta=1.5 
 candidate = [0.14514087 0.16997591 0.29876258 0.24921097 0.28231768 0.36282021] 
 mean = 2.667101 
 std = 0.109382 
 UCB = 2.831174 



In [12]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.12158283 0.28741647 0.32312591 0.23681484 0.27857032 0.496277  ]
mean = 2.7080194144593728
std = 0.04244029950794731


In [13]:
# --------------------------------------------------
# Final Function 7 Week 7 selection
# --------------------------------------------------
#
# The GP predicts a higher mean at the selected candidate
# than the current best observed output.
#
# The fitted ARD Matern kernel indicates that x4 and x5
# vary more rapidly than x2, x3 and x6, which have much
# longer fitted lengthscales.
#
# The highest GP predicted mean and UCB with beta=0.1
# selected the same candidate, while EI selected a nearby
# point with slightly higher uncertainty.
#
# I therefore use low-exploration UCB with beta=0.1.

beta = 0.1

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week7_candidate = candidates[final_idx]

print("Week 7 Function 7 candidate:")
print(week7_candidate)

print("\nPredicted mean:", mu[final_idx])
print("Predicted std:", sigma[final_idx])
print("UCB:", UCB[final_idx])

portal = "-".join(f"{x:.6f}" for x in week7_candidate)

print("\nPortal format:")
print(portal)

Week 7 Function 7 candidate:
[0.12158283 0.28741647 0.32312591 0.23681484 0.27857032 0.496277  ]

Predicted mean: 2.7080194144593728
Predicted std: 0.04244029950794731
UCB: 2.7122634444101674

Portal format:
0.121583-0.287416-0.323126-0.236815-0.278570-0.496277
